In [7]:
import os
import cv2
import numpy as np
from pathlib import Path

def evaluate_correction_with_deflect(img_without_deflect, img_with_deflect, img_transform, deflect):
    """
    Évalue uniquement la correction des pixels spécifiés dans le dictionnaire 'deflect'.
    Accepte des chemins de fichiers (str, Path) ou directement des matrices (np.ndarray).
    Retourne un score de 0% à 100%.
    """
    def load_image(img_input):
        # 1. Si c'est déjà une matrice numpy
        if isinstance(img_input, np.ndarray):
            return img_input.astype(np.float32, copy=False)
        # 2. Si c'est un chemin de fichier
        elif isinstance(img_input, (str, Path)):
            img_path = str(img_input)
            if not os.path.exists(img_path):
                raise FileNotFoundError(f"Image introuvable : {img_path}")
            return cv2.imread(img_path, -1).astype(np.float32, copy=False)
        # 3. Type non géré
        else:
            raise TypeError(f"Type non supporté : {type(img_input)}. Attendu : str, Path ou np.ndarray.")

    gt = load_image(img_without_deflect)      # 100% Original
    base = load_image(img_with_deflect)       # 0% Base défectueuse
    res = load_image(img_transform)           # Image à comparer

    score = []

    for x_key, y_intervals in deflect.items():
        x = int(x_key)
        for y_start, y_end in y_intervals:
            for y_coord in range(y_start, y_end):
                diff_ideale = float(gt[y_coord, x] - base[y_coord, x])
                
                # Sécurité indispensable pour éviter le RuntimeWarning (division par zéro)
                if diff_ideale != 0.0:
                    diff_reelle = float(res[y_coord, x] - base[y_coord, x])
                    pixel_score = (diff_reelle * 100.0) / diff_ideale
                    score.append(pixel_score)
                else:
                    # Si aucun défaut n'était présent sur ce pixel
                    if res[y_coord, x] == gt[y_coord, x]:
                        score.append(100.0)
                    else:
                        score.append(0.0)

    # Sécurité finale : évite un "nan" si la liste score est totalement vide
    return np.mean(score) if len(score) > 0 else 0.0